In [31]:
import sys
sys.path.insert(0, '../src')
from models import load_model
from utils import *
from tversky_utils import *

config = parse_config('../configs/tversky_proj.yaml')
# latent dim 6, fbank size 128

model = load_model(config, "../results/tversky_proj_gridrobot/tversky_proj_1784130650.pth")
data = load_data(config)
all_trajs = data["trajs"]
all_feats = data["features"]

loading data: gridrobot_1960


In [32]:
import numpy as np
import itertools
import torch

In [33]:
# using laptop (feature idx 0) because it is more query-able than upright, 
# see experiments/002-tversky-query-eval/figs/tversky_proj_fbank_size.png
hi_indices = np.where(all_feats[:,0] == np.max(all_feats[:,0]))[0]
lo_indices = np.where(all_feats[:,0] == np.min(all_feats[:,0]))[0]
print(f"{len(hi_indices)} hi indices, {len(lo_indices)} lo indices")
hi_trajs = all_trajs[hi_indices]
lo_trajs = all_trajs[lo_indices]

pairs = np.array(list(itertools.product(hi_indices, lo_indices)))    # all (max_traj, min_traj) pairs)
pairs.shape

56 hi indices, 448 lo indices


(25088, 2)

In [34]:
TRAIN_TEST_SPLIT = 0.8
n = len(pairs)
rng = np.random.default_rng(seed=0)
perm = rng.permutation(n)
n_train = int(n * TRAIN_TEST_SPLIT)
train_idx, test_idx = perm[:n_train], perm[n_train:]
train_pairs = pairs[train_idx]
test_pairs = pairs[test_idx]

In [35]:
TOP_FEATURE_COUNT = 128 # this is the feature bank size, so query will retrieve all salient features
TOP_RESULT_COUNT = 100
feature_bank = model.encoder[0].feature_bank.weight.detach()  # (F, D)
trajs_t = torch.as_tensor(all_trajs, dtype=torch.float32)
centered_trajs = (trajs_t - trajs_t.mean(0)).detach()       # (N, D)
def run_query(a_idx, b_idx, top_feature_count=TOP_FEATURE_COUNT, top_result_count=TOP_RESULT_COUNT):
    """s(a) - s(b): retrieve instances salient for a's features but not b's."""
    return retrieve_semantic_expression(
        instance_vectors=centered_trajs,
        feature_bank=feature_bank,
        expression=f"s({a_idx})-s({b_idx})",
        top_feature_count=top_feature_count,
        top_result_count=top_result_count,
    )



In [36]:
from collections import Counter
from scipy import stats

N_QUERIES = n_train
VOTE_FRAC = 0.5   # keep features seen in >= this fraction of queries so far
                  # 1.0 == intersection, ~0 == union

hi_votes, lo_votes = Counter(), Counter()
accs = []
for i in range(N_QUERIES):
    hi, lo = train_pairs[i]
    hi_res = run_query(hi, lo) # hi - lo
    lo_res = run_query(lo, hi) # lo - hi
    hi_votes.update(hi_res["semantic_features"])
    lo_votes.update(lo_res["semantic_features"])

    thresh = VOTE_FRAC * (i + 1)
    hi_feats = {f for f, c in hi_votes.items() if c >= thresh}
    lo_feats = {f for f, c in lo_votes.items() if c >= thresh}

    hi_top_instances = get_top_instances(centered_trajs, feature_bank, hi_feats, TOP_RESULT_COUNT)
    lo_top_instances = get_top_instances(centered_trajs, feature_bank, lo_feats, TOP_RESULT_COUNT)

    # t test of means (laptop only)
    hi_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in hi_top_instances]
    lo_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in lo_top_instances]
    # both_constant = np.std(hi_top_instances) == 0 and np.std(lo_top_instances) == 0
    # if len(a) >= 2 and len(b) >= 2 and not both_constant:
    t = stats.ttest_ind(hi_top_instances_true_feats, lo_top_instances_true_feats, alternative="greater")
    print(f"ttest pvalue after {i + 1} queries: {t.pvalue}  "
          f"(|hi_feats|={len(hi_feats)}, |lo_feats|={len(lo_feats)})")

ttest pvalue after 1 queries: 0.5210505941913648  (|hi_feats|=39, |lo_feats|=29)
ttest pvalue after 2 queries: 2.6082317062900967e-05  (|hi_feats|=63, |lo_feats|=51)
ttest pvalue after 3 queries: 1.3648456601763357e-10  (|hi_feats|=16, |lo_feats|=10)
ttest pvalue after 4 queries: 4.113605718206547e-06  (|hi_feats|=25, |lo_feats|=26)
ttest pvalue after 5 queries: 0.9834062862251693  (|hi_feats|=6, |lo_feats|=11)
ttest pvalue after 6 queries: 0.9631759340482593  (|hi_feats|=9, |lo_feats|=11)
ttest pvalue after 7 queries: 0.44579728895142934  (|hi_feats|=3, |lo_feats|=2)
ttest pvalue after 8 queries: 0.15323756891735257  (|hi_feats|=9, |lo_feats|=5)


RuntimeError: index_select(): Expected dtype int32 or int64 for index

```
ttest pvalue after 1 queries: 0.5210505941913648  (|hi_feats|=39, |lo_feats|=29)
ttest pvalue after 2 queries: 2.6082317062900967e-05  (|hi_feats|=63, |lo_feats|=51)
ttest pvalue after 3 queries: 1.3648456601763357e-10  (|hi_feats|=16, |lo_feats|=10)
```
^ this is promising... set a p value threshold (e.g. .001) and stop when it is reached, measure...
average number of queries before top 100 have significant difference in feature value
(should also try for tversky VAE)

then, train classifier

# num queries until significant:

In [39]:
from collections import Counter
from scipy import stats

N_QUERIES = n_train
MAX_QUERIES = 100
VOTE_FRAC = 0.5   # keep features seen in >= this fraction of queries so far
                  # 1.0 == intersection, ~0 == union
P_THRESHOLD = 0.05
N_SHUFFLES = 100

n_fails = 0
n_total_attempts = 0
n_queries_to_success = []

for j in range(N_SHUFFLES):
    # shuffle train pairs according to j
    rng = np.random.default_rng(j)
    shuffled_pairs = rng.permutation(train_pairs)

    hi_votes, lo_votes = Counter(), Counter()
    for i in range(MAX_QUERIES):
        hi, lo = shuffled_pairs[i]
        hi_res = run_query(hi, lo) # hi - lo
        lo_res = run_query(lo, hi) # lo - hi
        hi_votes.update(hi_res["semantic_features"])
        lo_votes.update(lo_res["semantic_features"])

        thresh = VOTE_FRAC * (i + 1)
        hi_feats = {f for f, c in hi_votes.items() if c >= thresh}
        lo_feats = {f for f, c in lo_votes.items() if c >= thresh}

        if not hi_feats or not lo_feats:
            print(f"fail after {i+1} queries: lo or hi feats is empty!")
            n_fails += 1
            break
        hi_top_instances = get_top_instances(centered_trajs, feature_bank, hi_feats, TOP_RESULT_COUNT)
        lo_top_instances = get_top_instances(centered_trajs, feature_bank, lo_feats, TOP_RESULT_COUNT)

        # t test of means (laptop only)
        hi_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in hi_top_instances]
        lo_top_instances_true_feats = [all_feats[inst["item_ix"], 0] for inst in lo_top_instances]

        t = stats.ttest_ind(hi_top_instances_true_feats, lo_top_instances_true_feats, alternative="greater")
        print(f"ttest pvalue after {i + 1} queries: {t.pvalue}  "
            f"(|hi_feats|={len(hi_feats)}, |lo_feats|={len(lo_feats)})")
        if t.pvalue <= P_THRESHOLD: 
            print(f"success after {i+1} queries!")
            n_queries_to_success.append(i+1)
            break
        if i == MAX_QUERIES - 1:
            print(f"failed after {i} queries")
            n_fails += 1
    n_total_attempts += 1

ttest pvalue after 1 queries: 0.9308983790486942  (|hi_feats|=20, |lo_feats|=43)
ttest pvalue after 2 queries: 0.7310794030313805  (|hi_feats|=53, |lo_feats|=68)
ttest pvalue after 3 queries: 0.327099365754638  (|hi_feats|=13, |lo_feats|=18)
ttest pvalue after 4 queries: 0.2580475756789387  (|hi_feats|=19, |lo_feats|=33)
ttest pvalue after 5 queries: 0.9930677271607072  (|hi_feats|=13, |lo_feats|=15)
ttest pvalue after 6 queries: 0.9999987492223467  (|hi_feats|=20, |lo_feats|=19)
ttest pvalue after 7 queries: 0.9641944277790444  (|hi_feats|=7, |lo_feats|=9)
ttest pvalue after 8 queries: 0.9977803357197469  (|hi_feats|=15, |lo_feats|=17)
ttest pvalue after 9 queries: 0.9823167297119378  (|hi_feats|=12, |lo_feats|=13)
ttest pvalue after 10 queries: 0.9828764024847028  (|hi_feats|=16, |lo_feats|=17)
ttest pvalue after 11 queries: 0.5862493645995648  (|hi_feats|=8, |lo_feats|=9)
ttest pvalue after 12 queries: 0.6276880970741616  (|hi_feats|=10, |lo_feats|=9)
ttest pvalue after 13 queries: 

In [40]:
print(n_fails)
print(n_total_attempts)
print(n_queries_to_success)

23
100
[1, 5, 4, 1, 5, 1, 5, 16, 1, 2, 3, 5, 5, 5, 5, 1, 1, 3, 2, 8, 3, 3, 4, 6, 3, 2, 1, 4, 2, 3, 12, 2, 1, 7, 2, 1, 1, 1, 3, 2, 1, 1, 1, 1, 2, 2, 19, 1, 6, 5, 1, 1, 2, 1, 4, 2, 1, 3, 5, 17, 3, 1, 3, 1, 8, 1, 2, 1, 1, 1, 1, 3, 13, 12, 1, 1, 1]


TODO try this for all tversky methods!

sweep:
* TOP_FEATURE_COUNT
* VOTE_FRAC

would be interesting to know: are there query orderings that make this more or less likely?